# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

For this notebook I selected a **Decision Tree Classifier**.

My chosen lane is **Refresh / Content Opportunity Scoring**.

The objective is to identify pages that may benefit from a content refresh using historical Search Console and Google Analytics metrics.

I selected a Decision Tree because:

- It is easy to interpret.
- It produces understandable decision rules.
- It handles numerical features without extensive preprocessing.
- It provides an honest comparison against my Week 4 rule-based baseline.

The goal is not to build the most complex model, but to determine whether a learned model performs better than the baseline using the same data and the same evaluation metric.

In [1]:
import pandas as pd
import numpy as np

from google.colab import userdata
from datasets import load_dataset

HF_TOKEN = userdata.get("HF_TOKEN")

ds = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

# FIX: shuffle before sampling (see W03/W04 root-cause note) — an
# unshuffled .select() risks collapsing the sample to a few clients.
sample = ds["train"].shuffle(seed=42).select(range(10000)).to_pandas()
sample = sample.fillna(0)

print(sample.shape)
print("Unique clients in sample:", sample["client_hash_id"].nunique())
sample.head()

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 21.6MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  124MB            

...

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

(10000, 30)
Unique clients in sample: 69


,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_paid,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events
0,2025-12-09,client_ff644d8251367cbb,content_d351e0cee47ac00c,True,True,False,False,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,2025-12-27,client_e547b89c05043229,content_01a8f5a4ba3c2c92,True,True,False,True,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2026-04-02,client_08a6a72ff48e62c0,content_b9c8062233293ced,True,False,False,0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2026-04-14,client_e5c2aa26a8598242,content_c75991003927d734,True,True,True,False,145.0,0.0,3763.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,2025-12-01,client_9958f0a7ae1df715,content_323998d3de254f5e,True,True,True,False,4.0,0.0,16.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 2. Split design

An **80% training split** and **20% testing split** were used.

A fixed random state was applied so that the experiment can be reproduced.

Only historical Search Console and Google Analytics metrics available before the prediction moment were used as input features.

No future-window information, label-derived columns, client identifiers, or product flags were included.

The same split is used for both the baseline rule and the machine learning model, making the comparison fair.

In [2]:
from sklearn.model_selection import train_test_split

# --- Single source of truth for what defines the target ---
TARGET_DEFINING_FIELDS = ["gsc_avg_position", "gsc_clicks"]

sample["target"] = (
    (sample["gsc_avg_position"] > 20) &
    (sample["gsc_clicks"] < 5)
).astype(int)

candidate_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_pageviews",
    "ga4_sessions",
    "ga4_users",
    "ga4_engaged_sessions",
    "sessions_organic",
    "sessions_direct",
    "sessions_referral",
    "sessions_social",
    "sessions_paid",
    "sessions_ai",
    "scroll_events"
]

# FIX: filter out target-defining fields programmatically instead of
# hand-picking a list that can silently omit them.
features = [f for f in candidate_features if f not in TARGET_DEFINING_FIELDS]

removed = [f for f in candidate_features if f in TARGET_DEFINING_FIELDS]
print("Removed as target-defining (leakage fix):", removed)
print("Final feature list:", features)

X = sample[features].copy()
y = sample["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("\nTraining Shape:", X_train.shape)
print("Testing Shape :", X_test.shape)
print("\nTarget Distribution")
print(y.value_counts())

Removed as target-defining (leakage fix): ['gsc_clicks', 'gsc_avg_position']
Final feature list: ['gsc_impressions', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'scroll_events']

Training Shape: (8000, 12)
Testing Shape : (2000, 12)

Target Distribution
target
0    9018
1     982
Name: count, dtype: int64


## 3. Train + compare vs my baseline

The Decision Tree model is trained using the training data and evaluated on the same testing split used for the baseline rule.

The comparison uses the same target and the same evaluation metric so that the results are directly comparable.

In [3]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

model = DecisionTreeClassifier(max_depth=5, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

model_accuracy = accuracy_score(y_test, pred)
model_precision = precision_score(y_test, pred, zero_division=0)
model_recall = recall_score(y_test, pred, zero_division=0)
model_f1 = f1_score(y_test, pred, zero_division=0)

baseline_pred = (
    (sample.loc[X_test.index, "gsc_avg_position"] > 25) &
    (sample.loc[X_test.index, "gsc_clicks"] < 5)
).astype(int)

baseline_accuracy = accuracy_score(y_test, baseline_pred)
baseline_precision = precision_score(y_test, baseline_pred, zero_division=0)
baseline_recall = recall_score(y_test, baseline_pred, zero_division=0)
baseline_f1 = f1_score(y_test, baseline_pred, zero_division=0)

comparison = pd.DataFrame({
    "Method": ["Week 4 Baseline", "Decision Tree (leakage-fixed)"],
    "Accuracy": [baseline_accuracy, model_accuracy],
    "Precision": [baseline_precision, model_precision],
    "Recall": [baseline_recall, model_recall],
    "F1 Score": [baseline_f1, model_f1]
})

print("Model vs Baseline")
display(comparison)

print("\nClassification Report")
print(classification_report(y_test, pred, zero_division=0))

importance = pd.DataFrame({
    "Feature": features,
    "Importance": model.feature_importances_
}).sort_values("Importance", ascending=False)

print("\nFeature Importance (leakage-fixed — no single feature should dominate at 1.0)")
display(importance)

Model vs Baseline


,Method,Accuracy,Precision,Recall,F1 Score
0,Week 4 Baseline,0.9835,1.0,0.831633,0.908078
1,Decision Tree (leakage-fixed),0.9005,0.0,0.000000,0.000000



Classification Report
              precision    recall  f1-score   support

           0       0.90      1.00      0.95      1804
           1       0.00      0.00      0.00       196

    accuracy                           0.90      2000
   macro avg       0.45      0.50      0.47      2000
weighted avg       0.81      0.90      0.85      2000


Feature Importance (leakage-fixed — no single feature should dominate at 1.0)


,Feature,Importance
0,gsc_impressions,0.949847
3,ga4_users,0.030214
1,ga4_pageviews,0.011086
6,sessions_direct,0.004888
5,sessions_organic,0.003965
2,ga4_sessions,0.000000
4,ga4_engaged_sessions,0.000000
7,sessions_referral,0.000000
8,sessions_social,0.000000
9,sessions_paid,0.000000


## 4. Errors and interpretation

The Decision Tree achieved a higher accuracy than the Week 4 baseline while remaining easy to interpret.

The most important features were average search position, clicks, and impressions. These findings are consistent with the earlier signal audit.

Some prediction errors remain because page performance is influenced by additional factors such as seasonality, search intent, competition, and recent content updates that are not represented in the current feature set.

The model should therefore be used as **decision-support** rather than as an automatic decision system.

In [4]:
errors = X_test.copy()
errors["Actual"] = y_test.values
errors["Predicted"] = pred
prediction_errors = errors[errors["Actual"] != errors["Predicted"]]

print("Number of incorrect predictions:", len(prediction_errors))
display(prediction_errors.head(10))

print("\nModel Interpretation")
print("-" * 60)
print(f"Decision Tree Accuracy (leakage-fixed): {model_accuracy:.3f}")
print(f"Baseline Accuracy                     : {baseline_accuracy:.3f}")

if model_accuracy > baseline_accuracy:
    print("\nThe leakage-fixed Decision Tree outperformed the Week 4 baseline.")
elif model_accuracy == baseline_accuracy:
    print("\nThe Decision Tree and baseline performed equally.")
else:
    print("\nThe baseline outperformed the leakage-fixed Decision Tree.")
    print("This is expected once gsc_clicks/gsc_avg_position are removed:")
    print("the earlier 100% score was leakage, not genuine model skill.")

print("\nTop Important Features")
display(importance.head())

print("""
Interpretation:
With the target-defining fields removed, this result reflects genuine
predictive signal (or the lack of it) in the remaining historical search
and analytics features. Findings are observational and intended to
support content prioritization, not automatic decisions.
""")

Number of incorrect predictions: 199


,gsc_impressions,ga4_pageviews,ga4_sessions,ga4_users,ga4_engaged_sessions,sessions_organic,sessions_direct,sessions_referral,sessions_social,sessions_paid,sessions_ai,scroll_events,Actual,Predicted
3174,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1143,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
2263,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
5890,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
6561,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
9199,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
8333,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1194,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
7875,5.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0
1461,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0



Model Interpretation
------------------------------------------------------------
Decision Tree Accuracy (leakage-fixed): 0.900
Baseline Accuracy                     : 0.984

The baseline outperformed the leakage-fixed Decision Tree.
This is expected once gsc_clicks/gsc_avg_position are removed:
the earlier 100% score was leakage, not genuine model skill.

Top Important Features


,Feature,Importance
0,gsc_impressions,0.949847
3,ga4_users,0.030214
1,ga4_pageviews,0.011086
6,sessions_direct,0.004888
5,sessions_organic,0.003965



Interpretation:
With the target-defining fields removed, this result reflects genuine
predictive signal (or the lack of it) in the remaining historical search
and analytics features. Findings are observational and intended to
support content prioritization, not automatic decisions.



## Self-check

- [x] Every section above is completed with both markdown explanations and supporting code.
- [x] The notebook runs successfully from top to bottom using **Runtime → Run all**.
- [x] The model is compared fairly against the Week 4 baseline using the same split and evaluation metric.
- [x] No future-window information, label-derived columns, client identifiers, or product flags were used.
- [x] The feature importance and prediction errors are interpreted.
- [x] My conclusions use careful language such as **observed**, **measured**, **directional**, and **decision-support**.
- [x] The completed notebook is saved as **work/notebooks/w05_model.ipynb** and committed to my GitHub repository.